<a href="https://colab.research.google.com/github/ingkapat/predictive-bottleneck/blob/main/dataset.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [19]:
# Cell 1: Install
!pip install openai tqdm nest_asyncio -q

In [20]:
# Cell 2: Mount Google Drive (ต้อง Allow permission)
from google.colab import drive
drive.mount('/content/drive')
print('Mount Drive สำเร็จ')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Mount Drive สำเร็จ


In [21]:
# Cell 3: Config
import os
import nest_asyncio
nest_asyncio.apply()

from google.colab import userdata
os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY')

CONFIG = {
    # scam (label=1) รวม ~10,200
    'per_scam_short':     900,  # 6 × 900  = 5,400
    'per_scam_long':      600,  # 8 × 600  = 4,800

    # not-scam (label=0) รวม ~12,000
    'per_general_chat':   500,  # 4 × 500  = 2,000
    'per_verification':   500,  # 3 × 500  = 1,500
    'per_official':       500,  # 4 × 500  = 2,000
    'per_hard_negative':  400,  # 7 × 400  = 2,800
    'per_unknown':        300,  # 4 × 300  = 1,200
    'per_delivery':       300,  # 4 × 300  = 1,200
    'per_legit_promo':    260,  # 5 × 260  = 1,300

    'min_turns': 1,
    'max_turns': 6,

    'model':       'gpt-4o-mini',
    'temperature': 1.0,
    'max_retries': 5,
    'concurrent':  7,

    # save ลง Google Drive โดยตรง
    'output_dir':  '/content/drive/MyDrive/scam_dataset',
    'output_file': 'dataset.jsonl',
}

os.makedirs(CONFIG['output_dir'], exist_ok=True)

SCAM_SHORT_SUBTYPES = [
    'reward', 'phishing', 'otp_hijack', 'sextortion', 'loan_scam', 'sms_alert',
]
SCAM_LONG_SUBTYPES = [
    'callcenter', 'investment', 'romance', 'tech_support',
    'job_scam', 'money_mule', 'impersonation', 'fake_police_addline',
]
GENERAL_CHAT_SUBTYPES   = ['friend_hangout', 'family_check', 'couple_talk', 'colleague']
VERIFICATION_SUBTYPES   = ['otp_thai', 'otp_english', 'reset_code']
OFFICIAL_SUBTYPES       = ['bank_real', 'gov_notice', 'hospital', 'company_hr']
HARD_NEGATIVE_SUBTYPES  = [
    'real_otp', 'real_job', 'friend_borrow', 'real_parcel',
    'friend_new_number', 'colleague_expense', 'family_help',
]
UNKNOWN_SUBTYPES      = ['silent_call', 'wrong_number', 'connection_issue', 'incomplete_speech']
DELIVERY_SUBTYPES     = ['kerry_courier', 'food_delivery', 'ems_post', 'lazada_shopee']
LEGIT_PROMO_SUBTYPES  = ['ais_promo', 'true_promo', 'dtac_promo', 'bank_offer', 'insurance_offer']

n_scam  = CONFIG['per_scam_short'] * len(SCAM_SHORT_SUBTYPES) + CONFIG['per_scam_long'] * len(SCAM_LONG_SUBTYPES)
n_not   = (CONFIG['per_general_chat'] * len(GENERAL_CHAT_SUBTYPES) +
           CONFIG['per_verification'] * len(VERIFICATION_SUBTYPES) +
           CONFIG['per_official']     * len(OFFICIAL_SUBTYPES) +
           CONFIG['per_hard_negative']* len(HARD_NEGATIVE_SUBTYPES) +
           CONFIG['per_unknown']      * len(UNKNOWN_SUBTYPES) +
           CONFIG['per_delivery']     * len(DELIVERY_SUBTYPES) +
           CONFIG['per_legit_promo']  * len(LEGIT_PROMO_SUBTYPES))
n_total = n_scam + n_not

print('Config พร้อม')
print(f'  scam     : {n_scam}')
print(f'  not-scam : {n_not}')
print(f'  รวม      : {n_total}')
print(f'  save ที่  : {CONFIG["output_dir"]}')

Config พร้อม
  scam     : 10200
  not-scam : 12000
  รวม      : 22200
  save ที่  : /content/drive/MyDrive/scam_dataset


In [22]:
# Cell 4: Patterns

SCAM_SHORT_PATTERNS = {
    'reward':     'Notification claiming user won iPhone/cash/gold prize. '
                  'Requests contact via Line, link click, or ID card photo.',
    'phishing':   'SMS/notification about account locked or parcel delivery failure. '
                  'Pushes user to click bit.ly link or verify credentials.',
    'otp_hijack': 'Caller impersonates bank, requests 6-digit OTP code for verification. '
                  'Will use OTP to drain account.',
    'sextortion': 'Threatens to release private photos/videos unless payment is sent.',
    'loan_scam':  'Offers loan approval without credit check. '
                  'Requires upfront fee or ID document submission.',
    'sms_alert':  'Claims tax refund or social security benefit. '
                  'Requires dialing USSD code or submitting personal data.',
}

SCAM_LONG_PATTERNS = {
    'callcenter':    'Call center scam impersonating police, DSI, or revenue department. '
                     'Claims account is linked to criminal case. Demands transfer to safe account. '
                     'Forbids hanging up or telling anyone.',
    'investment':    'Investment scam guaranteeing 30-50% monthly return with no risk. '
                     'Requests user to chat or click link to register and transfer funds urgently.',
    'romance':       'Romance scam with affectionate messages. '
                     'Claims to be foreign businessman, soldier, or doctor. '
                     'Eventually requests emergency loan transfer.',
    'tech_support':  'Impersonates Microsoft, Apple, or bank technical support. '
                     'Reports virus or hacked account. '
                     'Instructs user to install AnyDesk or TeamViewer for remote access.',
    'job_scam':      'Online job offer for liking videos or completing tasks. '
                     'Initially pays small amounts, then requires deposits to unlock withdrawals.',
    'money_mule':    'Offers commission for receiving and forwarding bank transfers. '
                     'Claims to be foreign company needing local Thai accounts.',
    'impersonation': 'Impersonates friend or relative claiming new phone number. '
                     'Creates emergency requiring urgent money transfer. '
                     'Refuses callbacks to old number. No personal context, vague identity.',
    'fake_police_addline':
                     'Impersonates police officer or government official. '
                     'Claims user is involved in case or has document needing verification. '
                     'Asks user to add Line ID like @police-dsi or @official-xxx.',
}

GENERAL_CHAT_PATTERNS = {
    'friend_hangout': 'Friend invites for shopping, food, or movies. Casual language with กู มึง แก เธอ.',
    'family_check':   'Family member checking on user, asking about meals or work. Warm tone.',
    'couple_talk':    'Partner calling to chat, set meeting, or express affection.',
    'colleague':      'Coworker discussing meetings, work tasks, or lunch plans.',
}

VERIFICATION_PATTERNS = {
    'otp_thai':    'Automated SMS/voice message in Thai delivering OTP code. '
                   'Includes warning not to share with anyone.',
    'otp_english': 'Automated message in English. Format: '
                   '"Your verification code is XXXXXX. Do not share."',
    'reset_code':  'Password reset confirmation code or 2FA code delivery.',
}

OFFICIAL_PATTERNS = {
    'bank_real':  'Bank notification about credit card statement, expiring card, or transaction confirmation. '
                  'Does not request OTP or sensitive information.',
    'gov_notice': 'Government office (revenue dept, social security, transport) announcing rights or appointments. '
                  'Does not request payment.',
    'hospital':   'Hospital calling for appointment scheduling, lab results, or rescheduling.',
    'company_hr': 'Real company HR scheduling job interview. Identifies company name and position clearly.',
}

HARD_NEGATIVE_PATTERNS = {
    'real_otp':      'Real bank requesting OTP for transaction confirmation. '
                     'Includes reference number. Does not ask for additional credentials.',
    'real_job':      'Real HR scheduling urgent interview because position closing soon. '
                     'Does not request payment or ID documents.',
    'friend_borrow': 'Close friend genuinely borrowing money. Uses informal pronouns กู/มึง. '
                     'Discusses other topics first. Provides personal context. '
                     'Does not pressure if user wants to verify.',
    'real_parcel':   'Real delivery service notifying about package. Includes tracking number. '
                     'Does not send links or request payment.',
    'friend_new_number':
                     'Real friend with new phone number asking to borrow small amount. '
                     'Provides clear personal context. Reason for new number is mundane. '
                     'Not pressuring. Willing to wait for user to verify.',
    'colleague_expense':
                     'Coworker asking to split bill or borrow small amount for lunch/coffee. '
                     'References specific workplace context. Casual tone. Small amount.',
    'family_help':
                     'Family member asking for financial help with real life situation. '
                     'References family events. Conversational pace, not pressuring.',
}

UNKNOWN_PATTERNS = {
    'silent_call':       'Caller does not speak. May only have background noise or breathing.',
    'wrong_number':      'Caller asks for someone who is not at this number. Brief exchange, hangs up.',
    'connection_issue':  'Phone connection has noise, echo, or breaking up. Cannot proceed.',
    'incomplete_speech': 'Caller says incomplete sentences, mumbles, or speech is cut off.',
}

DELIVERY_PATTERNS = {
    'kerry_courier':  'Kerry Express, Flash Express, J&T courier calling. Has tracking number. '
                      'Asks for delivery address or schedule. Does NOT ask for payment or links.',
    'food_delivery':  'Lineman, Grab, Foodpanda rider calling. Asks for location or to come down. '
                      'Does NOT ask for payment via transfer.',
    'ems_post':       'Thailand Post delivering EMS or registered mail. Has tracking number. '
                      'Does NOT request fee transfer.',
    'lazada_shopee':  'Lazada or Shopee customer service confirming order or delivery date. '
                      'Does NOT ask for payment info or OTP.',
}

LEGIT_PROMO_PATTERNS = {
    'ais_promo':        'AIS customer service offering package upgrade or promotion. '
                        'Mentions specific package name and price. Does NOT ask for OTP or transfer.',
    'true_promo':       'True Move H or TrueOnline calling about internet or mobile package. '
                        'Mentions specific speed/price. Does NOT ask for payment via link or OTP.',
    'dtac_promo':       'Dtac customer service offering package switch or 5G promotion. '
                        'Specific package details mentioned. Does NOT ask for sensitive data.',
    'bank_offer':       'Real bank offering credit card or increased credit limit to existing customer. '
                        'Does NOT ask for OTP, password, or to click any link.',
    'insurance_offer':  'Insurance company offering health/life/travel insurance plan. '
                        'Quotes premium and coverage. Does NOT ask for credit card details over the phone.',
}

VICTIM_STYLES = {
    'naive':     'Trusting and easily persuaded. Brief positive responses like อือ, อ๋อ, โอเค, จริงเหรอ.',
    'skeptical': 'Mildly suspicious. Asks clarifying questions like แบบว่า, เอ๊ะ, รอก่อนนะ, แน่ใจเหรอ.',
    'aware':     'Recognizes the scam pattern. Cuts off with เฮ้ย หยุดก่อน, ไม่เอาแล้ว, จะวางสายนะ.',
    'busy':      'Busy or in a hurry. Short responses like รีบหน่อย, แป๊บนึง, กำลังขับรถ, มีอะไรเร็วๆ.',
    'elderly':   'Older person, slow to respond. Says อะไรนะ, พูดอีกทีได้ไหม, ฟังไม่ค่อยชัด.',
}

LANGUAGE_STYLES = {
    'formal': 'Standard Thai with polite particles ครับ/ค่ะ.',
    'casual': 'Colloquial Thai with particles อ่ะ, นะ, จ้า, เอ้า, อือ. Short sentences.',
    'mixed':  'Mixed Thai-English code-switching. Words like OK, confirm, check, sure.',
}

OPENING_PHRASES = [
    'ฮัลโหล', 'สวัสดีครับ', 'สวัสดีค่ะ', 'ฮัลโหลครับ', 'ฮัลโหลค่ะ',
    'อ้าว', 'เฮ้ย', 'นี่', 'เห้', 'ไงเว้ย',
    'หวัดดีครับ', 'หวัดดีค่ะ', 'อ้าวสวัสดี',
    'ขอโทษครับ', 'ขออนุญาตครับ', 'รบกวนหน่อยครับ',
    'พี่ครับ', 'น้องครับ', 'คุณครับ', 'พี่จ๋า',
    '(เข้าเรื่องทันที ไม่มีคำเปิด)',
]

print('โหลด patterns สำเร็จ')

โหลด patterns สำเร็จ


In [23]:
# Cell 5: System prompt + prompt builder
import random

SYSTEM_PROMPT = '''You are a Thai language conversation writer specializing in realistic phone dialogues.
Your task is to generate synthetic phone conversations for training a scam detection AI.

Critical requirement: Generated dialogue must sound like authentic Thai phone conversations,
not formal written language.

Guidelines:
1. Use realistic spoken Thai: ครับ, ค่ะ, นะ, อ่ะ, เว้ย, จ้า, อือ, อ๋อ, เฮ้ย, อ้าว, แบบว่า, คือ
2. For close friends and family, use informal pronouns: กู, มึง, เอ็ง, แก, เธอ, ไอ้..., อี่...
3. Vary the opening phrase based on relationship and context.
4. Keep utterances short and natural.
5. Scammers tend to use overly formal language: กรุณา, โปรด, เรียน.
6. The user (call recipient) typically speaks less than the caller.
7. Some conversations may have only the caller speaking (broadcast/SMS style).

OUTPUT FORMAT (CRITICAL):
Return JSON object with EXACTLY this structure:
{"turns": [{"speaker": "caller", "text": "..."}, {"speaker": "user", "text": "..."}]}
- Key MUST be "turns"
- speaker value MUST be "caller" or "user" only
- text MUST be non-empty string
- No other keys, no text outside JSON'''

TURNS_HINT = '{"turns": [{"speaker": "caller", "text": "..."}]}'

def build_prompt(category, subtype, victim_style, lang_style, opening, min_t, max_t, extra=''):
    victim_section  = f'Recipient profile: {victim_style}\n' if victim_style else ''
    opening_section = f'Opening phrase hint: {opening}\n' if opening else ''
    return f'''Generate one phone conversation:

Category: {category}
Subtype: {subtype}
{victim_section}Language style: {lang_style}
{opening_section}Target length: {min_t}-{max_t} turns
{extra}

Required output format: {TURNS_HINT}
Use "speaker" key with value "caller" or "user" only.'''

print('Prompt builder พร้อม')

Prompt builder พร้อม


In [24]:
# Cell 6: API helpers (async + robust parsing)
from openai import AsyncOpenAI
import json, asyncio

client = AsyncOpenAI()

async def call_gpt_async(prompt, max_tokens=1000):
    response = await client.chat.completions.create(
        model=CONFIG['model'],
        temperature=CONFIG['temperature'],
        max_tokens=max_tokens,
        response_format={'type': 'json_object'},
        messages=[
            {'role': 'system', 'content': SYSTEM_PROMPT},
            {'role': 'user',   'content': prompt},
        ],
    )
    return response.choices[0].message.content.strip()

def normalize_speaker(s):
    if not s:
        return None
    s = str(s).strip().lower()
    if s in ['caller', 'call', 'a', 'agent', 'scammer', 'sender', 'speaker_a']:
        return 'caller'
    if s in ['user', 'recipient', 'b', 'victim', 'receiver', 'me', 'speaker_b', 'customer']:
        return 'user'
    return None

def parse_turns(text):
    data = json.loads(text)
    turns = None
    if isinstance(data, list):
        turns = data
    elif isinstance(data, dict):
        for key in ['turns', 'conversation', 'dialogue', 'messages', 'utterances']:
            if key in data and isinstance(data[key], list):
                turns = data[key]
                break
        if turns is None:
            for v in data.values():
                if isinstance(v, list):
                    turns = v
                    break
    if not turns:
        raise ValueError('No valid turns list')
    cleaned = []
    for t in turns:
        if not isinstance(t, dict):
            continue
        speaker = None
        for sk in ['speaker', 'role', 'speaker_name', 'who', 'from']:
            if sk in t:
                speaker = normalize_speaker(t[sk])
                if speaker:
                    break
        text_val = None
        for tk in ['text', 'content', 'message', 'utterance', 'dialogue']:
            if tk in t and isinstance(t[tk], str) and t[tk].strip():
                text_val = t[tk].strip()
                break
        if speaker and text_val:
            cleaned.append({'speaker': speaker, 'text': text_val})
    if len(cleaned) < 1:
        raise ValueError('No valid turns after cleaning')
    return cleaned

async def generate_one_async(task):
    prompt = build_prompt(
        category=task['category'],
        subtype=task['desc'],
        victim_style=task.get('victim_desc'),
        lang_style=task['lang_desc'],
        opening=task.get('opening'),
        min_t=task['min_t'],
        max_t=task['max_t'],
        extra=task.get('extra', ''),
    )
    for attempt in range(CONFIG['max_retries']):
        try:
            raw   = await call_gpt_async(prompt)
            turns = parse_turns(raw)
            return {
                'label':    task['label'],
                'category': task['category_simple'],
                'subtype':  task['subtype_key'],
                'turns':    turns,
            }
        except Exception:
            await asyncio.sleep(2 ** attempt)
    return None

# ทดสอบ API
async def test_api():
    raw    = await call_gpt_async('Return: {"turns": [{"speaker": "caller", "text": "test"}]}', max_tokens=100)
    parsed = parse_turns(raw)
    return parsed

result = await test_api()
print(f'API พร้อม: {result}')

API พร้อม: [{'speaker': 'caller', 'text': 'เฮ้ย! กูมีข่าวดีนะ แก!'}, {'speaker': 'user', 'text': 'อะไรอีกวะ?'}]


In [25]:
# Cell 7: สร้าง task list
import itertools

random.seed(42)

victim_keys = list(VICTIM_STYLES.keys())
lang_keys   = list(LANGUAGE_STYLES.keys())
matrix      = list(itertools.product(victim_keys, lang_keys))

def random_opening():
    return random.choice(OPENING_PHRASES)

tasks = []

for st in SCAM_SHORT_SUBTYPES:
    for i in range(CONFIG['per_scam_short']):
        v_key, l_key = matrix[i % len(matrix)]
        tasks.append({'label':1,'category':f'Scam (short) - {st}','category_simple':'scam',
                      'subtype_key':st,'desc':SCAM_SHORT_PATTERNS[st],
                      'victim_desc':VICTIM_STYLES[v_key],'lang_desc':LANGUAGE_STYLES[l_key],
                      'opening':random_opening(),'min_t':1,'max_t':3})

for st in SCAM_LONG_SUBTYPES:
    for i in range(CONFIG['per_scam_long']):
        v_key, l_key = matrix[i % len(matrix)]
        tasks.append({'label':1,'category':f'Scam (long) - {st}','category_simple':'scam',
                      'subtype_key':st,'desc':SCAM_LONG_PATTERNS[st],
                      'victim_desc':VICTIM_STYLES[v_key],'lang_desc':LANGUAGE_STYLES[l_key],
                      'opening':random_opening(),'min_t':3,'max_t':6})

for st in GENERAL_CHAT_SUBTYPES:
    for i in range(CONFIG['per_general_chat']):
        _, l_key = matrix[i % len(matrix)]
        tasks.append({'label':0,'category':'General chat','category_simple':'general_chat',
                      'subtype_key':st,'desc':GENERAL_CHAT_PATTERNS[st],
                      'lang_desc':LANGUAGE_STYLES[l_key],'opening':random_opening(),
                      'min_t':2,'max_t':5,'extra':'Use informal pronouns where appropriate.'})

for st in VERIFICATION_SUBTYPES:
    for _ in range(CONFIG['per_verification']):
        tasks.append({'label':0,'category':'Verification code','category_simple':'verification',
                      'subtype_key':st,'desc':VERIFICATION_PATTERNS[st],
                      'lang_desc':'Automated formal style',
                      'min_t':1,'max_t':1,'extra':'Caller-only message. No user response.'})

for st in OFFICIAL_SUBTYPES:
    for i in range(CONFIG['per_official']):
        _, l_key = matrix[i % len(matrix)]
        tasks.append({'label':0,'category':'Official call','category_simple':'official',
                      'subtype_key':st,'desc':OFFICIAL_PATTERNS[st],
                      'lang_desc':LANGUAGE_STYLES[l_key],'opening':random_opening(),
                      'min_t':1,'max_t':4,'extra':'Must not request OTP, payment, or sensitive data.'})

for st in HARD_NEGATIVE_SUBTYPES:
    for i in range(CONFIG['per_hard_negative']):
        _, l_key = matrix[i % len(matrix)]
        extra = ('Use informal pronouns. Discuss unrelated topics first.' if st == 'friend_borrow' else
                 'Caller must include specific personal context. Not pressuring.' if st == 'friend_new_number' else
                 'Reference specific workplace context. Small amount only.' if st == 'colleague_expense' else
                 'Reference family-specific context. Conversational pace.' if st == 'family_help' else '')
        tasks.append({'label':0,'category':f'Hard negative - {st}','category_simple':'hard_negative',
                      'subtype_key':st,'desc':HARD_NEGATIVE_PATTERNS[st],
                      'lang_desc':LANGUAGE_STYLES[l_key],'opening':random_opening(),
                      'min_t':2,'max_t':5,'extra':extra})

for st in UNKNOWN_SUBTYPES:
    for _ in range(CONFIG['per_unknown']):
        tasks.append({'label':0,'category':f'Unknown - {st}','category_simple':'unknown',
                      'subtype_key':st,'desc':UNKNOWN_PATTERNS[st],'lang_desc':'Variable',
                      'min_t':1,'max_t':3,'extra':'Content should be brief, unclear, or interrupted.'})

for st in DELIVERY_SUBTYPES:
    for i in range(CONFIG['per_delivery']):
        _, l_key = matrix[i % len(matrix)]
        tasks.append({'label':0,'category':f'Delivery - {st}','category_simple':'delivery',
                      'subtype_key':st,'desc':DELIVERY_PATTERNS[st],
                      'lang_desc':LANGUAGE_STYLES[l_key],'opening':random_opening(),
                      'min_t':2,'max_t':4,'extra':'Has tracking number. Does NOT request payment via transfer.'})

for st in LEGIT_PROMO_SUBTYPES:
    for i in range(CONFIG['per_legit_promo']):
        _, l_key = matrix[i % len(matrix)]
        tasks.append({'label':0,'category':f'Legit promo - {st}','category_simple':'legit_promo',
                      'subtype_key':st,'desc':LEGIT_PROMO_PATTERNS[st],
                      'lang_desc':LANGUAGE_STYLES[l_key],'opening':random_opening(),
                      'min_t':3,'max_t':6,'extra':'Mention specific package details. Does NOT ask for OTP or transfer.'})

random.shuffle(tasks)

from collections import Counter
label_dist = Counter(t['label'] for t in tasks)
print(f'จำนวน tasks ทั้งหมด: {len(tasks)}')
print(f'  label=1: {label_dist[1]} ({label_dist[1]/len(tasks)*100:.1f}%)')
print(f'  label=0: {label_dist[0]} ({label_dist[0]/len(tasks)*100:.1f}%)')

จำนวน tasks ทั้งหมด: 22200
  label=1: 10200 (45.9%)
  label=0: 12000 (54.1%)


In [26]:
import nest_asyncio
nest_asyncio.apply()

In [ ]:

# Cell 8: Generate (concurrent + resume support)
from tqdm.notebook import tqdm

DATASET_PATH = os.path.join(CONFIG['output_dir'], CONFIG['output_file'])

# Resume: นับ records ที่มีอยู่แล้วใน Drive
done_count = 0
if os.path.exists(DATASET_PATH):
    with open(DATASET_PATH, encoding='utf-8') as f:
        done_count = sum(1 for _ in f)
    print(f'Resume: มี {done_count} records แล้ว ข้าม...')

remaining_tasks = tasks[done_count:]
print(f'จะสร้างเพิ่มอีก: {len(remaining_tasks)} records')
print()

semaphore = asyncio.Semaphore(CONFIG['concurrent'])
lock      = asyncio.Lock()
successful = 0
failed     = 0

f_out = open(DATASET_PATH, 'a', encoding='utf-8')
pbar  = tqdm(total=len(remaining_tasks), desc='Generating')

async def process_task(task):
    global successful, failed
    async with semaphore:
        record = await generate_one_async(task)
    async with lock:
        if record is None:
            failed += 1
        else:
            f_out.write(json.dumps(record, ensure_ascii=False) + '\n')
            f_out.flush()
            os.fsync(f_out.fileno())  # เพิ่มบรรทัดนี้
            successful += 1
        pbar.update(1)
        pbar.set_postfix({'ok': successful, 'fail': failed})

await asyncio.gather(*[process_task(t) for t in remaining_tasks])

f_out.close()
pbar.close()

print()
print('Generation เสร็จ')
print(f'  สำเร็จ : {successful}')
print(f'  ล้มเหลว: {failed}')
print(f'  ไฟล์   : {DATASET_PATH}')

Resume: มี 7860 records แล้ว ข้าม...
จะสร้างเพิ่มอีก: 14340 records



Generating:   0%|          | 0/14340 [00:00<?, ?it/s]

In [ ]:
# Cell 9: สถิติ
records = []
with open(DATASET_PATH, encoding='utf-8') as f:
    for line in f:
        records.append(json.loads(line))

n_total = len(records)
n_scam  = sum(1 for r in records if r['label'] == 1)
n_legit = n_total - n_scam

print(f'สถิติ dataset')
print(f'  จำนวนทั้งหมด: {n_total}')
print(f'  scam        : {n_scam} ({n_scam/n_total*100:.1f}%)')
print(f'  not-scam    : {n_legit} ({n_legit/n_total*100:.1f}%)')
print()
print('แจกแจงตาม subtype:')
sub_counts = Counter(r.get('subtype', 'unknown') for r in records)
for sub, count in sorted(sub_counts.items()):
    print(f'  {sub:25s}: {count}')

In [ ]:
# Cell 10: ดูตัวอย่างสุ่ม
import random as rd
rd.seed(0)

for idx in rd.sample(range(len(records)), 5):
    r = records[idx]
    label_text = 'SCAM' if r['label'] == 1 else 'NOT_SCAM'
    print('-' * 60)
    print(f'label: {label_text} | subtype: {r.get("subtype")}')
    for t in r['turns']:
        print(f'  [{t["speaker"]:6s}] {t["text"]}')
    print()